Segment 1 :
Data Exploration & Filtering

Task 1 : Customer Profile & Email Domain. Show customers with signup date, segment, and extracted email domain.

In [ ]:
SELECT
    c.FIRST_NAME || ' ' || c.LAST_NAME AS customer_name,
    c.SIGNUP_DATE,
    c.SEGMENT,
    SPLIT_PART(c.EMAIL, '@', 2) AS email_domain
FROM
    DEMO_DW.SALES.DIM_CUSTOMER c;

Result: Customer details with extracted email domain were successfully retrieved.

Task 2: Active Products by Category. Show all active products sorted by category and descending unit price.

In [ ]:
SELECT
    p.PRODUCT_NAME,
    p.CATEGORY,
    p.UNIT_PRICE
FROM
    DEMO_DW.SALES.DIM_PRODUCT p
WHERE
    p.ACTIVE = TRUE
ORDER BY
    p.CATEGORY ASC,
    p.UNIT_PRICE DESC;

Result: Active products were listed and sorted by category and price.

Task 3: Active Promotions. Find all promotions that are currently active.

In [ ]:
SELECT
    p.PROMO_TYPE,
    p.START_DATE,
    p.END_DATE
FROM
    DEMO_DW.SALES.DIM_PROMOTION p
WHERE
    CURRENT_DATE >= p.START_DATE
    AND CURRENT_DATE <= p.END_DATE;

Result: No active promotions for the current date.

Task 4: Stores in Alberta & Ontario.Find all stores located in Alberta or Ontario.

In [ ]:
SELECT
    s.STORE_NAME,
    s.REGION
FROM
    DEMO_DW.SALES.DIM_STORE s
WHERE
    s.REGION IN ('AB', 'ON');

Result: No matching records were found for stores in Alberta (AB) or Ontario (ON), suggesting either missing data or different region coding in the dataset.

Task 5: First 10 Dates. Show the first 10 days from DIM_DATE ordered by FULL_DATE.

In [ ]:
SELECT
    d.FULL_DATE
FROM
    DEMO_DW.SALES.DIM_DATE d
ORDER BY
    d.FULL_DATE ASC
LIMIT
    10;

Result: First 10 dates were successfully retrieved from the date dimension.

Segment 2 : Aggregations and KPIs.

Task 1: Sales Performance Metrics Overview.
Calculate total revenue, total quantity, and average discount from V_SALES_ENRICHED.

In [ ]:
SELECT
    SUM(QTY * UNIT_PRICE) AS total_revenue,
    SUM(QTY) AS total_quantity,
    AVG(DISCOUNT_PCT) AS avg_discount
FROM
    DEMO_DW.SALES.V_SALES_ENRICHED;

Result: Key sales KPIs (revenue, quantity, discount) were successfully calculated.

-- Task 2

--2.1. Average Price per Category

In [ ]:
SELECT
    p.CATEGORY,
    AVG(p.UNIT_PRICE) AS avg_price
FROM
    DEMO_DW.SALES.DIM_PRODUCT p
GROUP BY
    p.CATEGORY;

Result: The analysis shows variation in average prices across product categories.

-- 2.2.
Product Count per Subcategory

In [ ]:
SELECT
    p.SUBCATEGORY,
    COUNT(*) AS product_count
FROM
    DEMO_DW.SALES.DIM_PRODUCT p
GROUP BY
    p.SUBCATEGORY;

Result: Product count per subcategory calculated successfully.

-- Task 3

-- Top 3 Categories with the highest total revenue

In [ ]:
SELECT
    CATEGORY,
    SUM(QTY * UNIT_PRICE) AS total_revenue
FROM
    DEMO_DW.SALES.V_SALES_ENRICHED
GROUP BY
    CATEGORY
ORDER BY
    total_revenue DESC
LIMIT
    3;

Result: The query shows the top 3 product categories with the highest total revenue.

-- Task 4

-- Promotion Performance type generates the highest average revenue per order

In [ ]:
SELECT
    p.PROMO_TYPE,
    AVG(f.QTY * f.UNIT_PRICE) AS avg_revenue_per_order
FROM
    DEMO_DW.SALES.FACT_SALES f
    JOIN DEMO_DW.SALES.DIM_PROMOTION p ON f.PROMO_KEY = p.PROMO_KEY
GROUP BY
    p.PROMO_TYPE
ORDER BY
    avg_revenue_per_order DESC
LIMIT
    1;

Result: The analysis highlights the most effective promotion type based on average revenue per order.

-- Task 5

-- Customers per Segment in each segment sorted from most to least

In [ ]:
SELECT
    SEGMENT,
    COUNT(*) AS customer_count
FROM
    DEMO_DW.SALES.DIM_CUSTOMER
GROUP BY
    SEGMENT
ORDER BY
    customer_count DESC;

Result: The query shows the number of customers in each segment, ordered from most to least.

Segment 3 : Joins Practice

-- Task: 1

-- Combine tables to show customer, product, store, and revenue

In [ ]:
SELECT
    c.FIRST_NAME || ' ' || c.LAST_NAME AS customer_name,
    p.PRODUCT_NAME,
    s.STORE_NAME,
    fs.QTY * fs.UNIT_PRICE AS revenue
FROM
    DEMO_DW.SALES.FACT_SALES fs
    JOIN DEMO_DW.SALES.DIM_CUSTOMER c ON fs.CUSTOMER_KEY = c.CUSTOMER_KEY
    JOIN DEMO_DW.SALES.DIM_PRODUCT p ON fs.PRODUCT_KEY = p.PRODUCT_KEY
    JOIN DEMO_DW.SALES.DIM_STORE s ON fs.STORE_KEY = s.STORE_KEY;

Result: The combined dataset provides a complete view of customer purchases, including product, store, and revenue details.

-- Task: 2

-- Find customers who never made a purchase

In [ ]:
SELECT
    c.FIRST_NAME || ' ' || c.LAST_NAME AS customer_name
FROM
    DEMO_DW.SALES.DIM_CUSTOMER c
    LEFT JOIN DEMO_DW.SALES.FACT_SALES s ON c.CUSTOMER_KEY = s.CUSTOMER_KEY
WHERE
    s.CUSTOMER_KEY IS NULL;

Result: No customers found without purchases.

-- Task: 3

-- Find products that have no sales records

In [ ]:
SELECT
    p.PRODUCT_NAME
FROM
    DEMO_DW.SALES.DIM_PRODUCT p
    LEFT JOIN DEMO_DW.SALES.FACT_SALES s ON p.PRODUCT_KEY = s.PRODUCT_KEY
WHERE
    s.PRODUCT_KEY IS NULL;

Result: No products without sales were found, indicating that all products have at least one recorded transaction.

-- Task: 4

-- List all entities with a label indicating type

In [ ]:
SELECT
    c.FIRST_NAME || ' ' || c.LAST_NAME AS name,
    'Customer' AS EntityType
FROM
    DEMO_DW.SALES.DIM_CUSTOMER c
UNION
SELECT
    s.STORE_NAME AS name,
    'Store' AS EntityType
FROM
    DEMO_DW.SALES.DIM_STORE s;

Result: The analysis creates a unified view of entities by combining customers and stores into a single dataset using UNION.

-- Task:5

-- Show total sales and number of unique customers per store

In [ ]:
SELECT
    s.STORE_NAME,
    SUM(fs.QTY * fs.UNIT_PRICE) AS total_sales,
    COUNT(DISTINCT fs.CUSTOMER_KEY) AS unique_customers
FROM
    DEMO_DW.SALES.FACT_SALES fs
    JOIN DEMO_DW.SALES.DIM_STORE s ON fs.STORE_KEY = s.STORE_KEY
GROUP BY
    s.STORE_NAME
ORDER BY
    total_sales DESC;

Result: The query shows total sales and the number of unique customers for each store.

Segment 4 : Conditional Logic & Filtering

--Task:1

-- Label revenue as High, Medium, or Low

In [ ]:
SELECT
    fs.CUSTOMER_KEY,
    fs.PRODUCT_KEY,
    fs.QTY * fs.UNIT_PRICE AS revenue,
    CASE
        WHEN fs.QTY * fs.UNIT_PRICE > 100 THEN 'High'
        WHEN fs.QTY * fs.UNIT_PRICE BETWEEN 50
        AND 100 THEN 'Medium'
        ELSE 'Low'
    END AS revenue_category
FROM
    DEMO_DW.SALES.FACT_SALES fs;

Result: The analysis segments revenue into performance levels, enabling easier interpretation of sales distribution.

-- Task: 2

-- Find customers whose emails contain gmail.com

In [ ]:
SELECT
    c.FIRST_NAME,
    c.LAST_NAME,
    c.EMAIL
FROM
    DEMO_DW.SALES.DIM_CUSTOMER c
WHERE
    c.EMAIL LIKE '%gmail.com%';

Result: The dataset does not contain any customers with gmail.com email addresses.

-- Task: 3

-- Find products that start with "S" or end with "er"

In [ ]:
SELECT
    p.PRODUCT_NAME
FROM
    DEMO_DW.SALES.DIM_PRODUCT p
WHERE
    p.PRODUCT_NAME LIKE 'S%'
    OR p.PRODUCT_NAME LIKE '%er';

Result: Products above the average price were successfully identified using a subquery.

-- Task 4

-- Find products more expensive than the average unit price

In [ ]:
SELECT
    PRODUCT_NAME,
    UNIT_PRICE
FROM
    DEMO_DW.SALES.DIM_PRODUCT
WHERE
    UNIT_PRICE > (
        SELECT
            AVG(UNIT_PRICE)
        FROM
            DEMO_DW.SALES.DIM_PRODUCT
    );

Result: The analysis identifies products that are more expensive than the average unit price, highlighting higher-value items.


-- Task 5

-- Show customers whose total revenue is above average

In [ ]:
SELECT
    c.FIRST_NAME || ' ' || c.LAST_NAME AS customer_name,
    SUM(fs.QTY * fs.UNIT_PRICE) AS total_revenue
FROM
    DEMO_DW.SALES.FACT_SALES fs
    JOIN DEMO_DW.SALES.DIM_CUSTOMER c ON fs.CUSTOMER_KEY = c.CUSTOMER_KEY
GROUP BY
    c.FIRST_NAME,
    c.LAST_NAME
HAVING
    SUM(fs.QTY * fs.UNIT_PRICE) > (
        SELECT
            AVG(customer_revenue)
        FROM
            (
                SELECT
                    SUM(QTY * UNIT_PRICE) AS customer_revenue
                FROM
                    DEMO_DW.SALES.FACT_SALES
                GROUP BY
                    CUSTOMER_KEY
            ) sub
    );

Result: The analysis identifies high-value customers whose total revenue is above the overall average.



Segment: 5
Window Functions

-- Task 1

-- Show total revenue and average order value per customer

In [ ]:
SELECT
    c.FIRST_NAME || ' ' || c.LAST_NAME AS customer_name,
    fs.QTY * fs.UNIT_PRICE AS order_value,
    SUM(fs.QTY * fs.UNIT_PRICE) OVER(PARTITION BY fs.CUSTOMER_KEY) AS total_revenue,
    AVG(fs.QTY * fs.UNIT_PRICE) OVER(PARTITION BY fs.CUSTOMER_KEY) AS avg_order_value
FROM
    DEMO_DW.SALES.FACT_SALES fs
    JOIN DEMO_DW.SALES.DIM_CUSTOMER c ON fs.CUSTOMER_KEY = c.CUSTOMER_KEY;

Result: Customer-level revenue metrics were calculated using window functions.

-- Task 2

-- Rank products based on revenue within each category

In [ ]:
SELECT
    p.CATEGORY,
    p.PRODUCT_NAME,
    SUM(fs.QTY * fs.UNIT_PRICE) AS total_revenue,
    RANK() OVER (
        PARTITION BY p.CATEGORY
        ORDER BY
            SUM(fs.QTY * fs.UNIT_PRICE) DESC
    ) AS product_rank
FROM
    DEMO_DW.SALES.FACT_SALES fs
    JOIN DEMO_DW.SALES.DIM_PRODUCT p ON fs.PRODUCT_KEY = p.PRODUCT_KEY
GROUP BY
    p.CATEGORY,
    p.PRODUCT_NAME;

Result: Products were successfully ranked by revenue within each category.

-- Task3

-- Calculate cumulative revenue over time

In [ ]:
SELECT
    d.FULL_DATE,
    SUM(fs.QTY * fs.UNIT_PRICE) AS daily_revenue,
    SUM(SUM(fs.QTY * fs.UNIT_PRICE)) OVER (
        ORDER BY
            d.FULL_DATE
    ) AS running_total
FROM
    DEMO_DW.SALES.FACT_SALES fs
    JOIN DEMO_DW.SALES.DIM_DATE d ON fs.DATE_KEY = d.DATE_KEY
GROUP BY
    d.FULL_DATE
ORDER BY
    d.FULL_DATE;

Result: Running total of daily revenue calculated successfully using window functions.

-- Task 4

-- Show how each sale differs from the average revenue

In [ ]:
SELECT
    revenue,
    avg_revenue,
    revenue - avg_revenue AS revenue_diff
FROM
    (
        SELECT
            fs.QTY * fs.UNIT_PRICE AS revenue,
            AVG(fs.QTY * fs.UNIT_PRICE) OVER() AS avg_revenue
        FROM
            DEMO_DW.SALES.FACT_SALES fs
    ) t;

Result: Revenue differences from the average were calculated successfully.

-- Task 5

-- Find the first order for each customer

In [ ]:
SELECT
    customer_name,
    FULL_DATE,
    revenue
FROM
    (
        SELECT
            c.FIRST_NAME || ' ' || c.LAST_NAME AS customer_name,
            d.FULL_DATE,
            fs.QTY * fs.UNIT_PRICE AS revenue,
            ROW_NUMBER() OVER (
                PARTITION BY fs.CUSTOMER_KEY
                ORDER BY
                    d.FULL_DATE
            ) AS row_num
        FROM
            DEMO_DW.SALES.FACT_SALES fs
            JOIN DEMO_DW.SALES.DIM_CUSTOMER c ON fs.CUSTOMER_KEY = c.CUSTOMER_KEY
            JOIN DEMO_DW.SALES.DIM_DATE d ON fs.DATE_KEY = d.DATE_KEY
    ) t
WHERE
    row_num = 1;

Result: The first order for each customer was successfully identified using ROW_NUMBER().

Segment : 6 Advanced SQL Challenge

--Task 1

--Top 5 Customers Contribution to Revenue

In [ ]:
SELECT
    c.FIRST_NAME || ' ' || c.LAST_NAME AS customer_name,
    SUM(fs.QTY * fs.UNIT_PRICE) AS total_revenue
FROM
    DEMO_DW.SALES.FACT_SALES fs
    JOIN DEMO_DW.SALES.DIM_CUSTOMER c ON fs.CUSTOMER_KEY = c.CUSTOMER_KEY
GROUP BY
    c.FIRST_NAME,
    c.LAST_NAME
ORDER BY
    total_revenue DESC
LIMIT
    5;

Result: The top 5 customers were identified based on total revenue. These customers contribute the highest share of sales and are key contributors to overall business performance.

-- Task 2

-- Analysis of Discount Impact by Promotion Type

In [ ]:
SELECT
    p.PROMO_TYPE,
    SUM(fs.QTY * fs.UNIT_PRICE * fs.DISCOUNT_PCT) AS total_discount_amount
FROM
    DEMO_DW.SALES.FACT_SALES fs
    JOIN DEMO_DW.SALES.DIM_PROMOTION p ON fs.PROMO_KEY = p.PROMO_KEY
GROUP BY
    p.PROMO_TYPE
ORDER BY
    total_discount_amount DESC;

Result: The analysis shows the promotion type that contributed the highest total discount amount, indicating the most impactful discount strategy.

-- Task 3

-- Top Performing Store-Product Combination by Revenue

In [ ]:
SELECT
    st.STORE_NAME,
    p.PRODUCT_NAME,
    SUM(fs.QTY * fs.UNIT_PRICE) AS total_revenue
FROM
    DEMO_DW.SALES.FACT_SALES fs
    JOIN DEMO_DW.SALES.DIM_STORE st ON fs.STORE_KEY = st.STORE_KEY
    JOIN DEMO_DW.SALES.DIM_PRODUCT p ON fs.PRODUCT_KEY = p.PRODUCT_KEY
GROUP BY
    st.STORE_NAME,
    p.PRODUCT_NAME
ORDER BY
    total_revenue DESC
LIMIT
    1;

Result: The analysis shows the top-performing store and product combination that generated the highest total revenue, highlighting the most profitable sales pair.

-- Task 4

-- Show total revenue by year and month

In [ ]:
SELECT
    d.YEAR,
    d.MONTH,
    SUM(fs.QTY * fs.UNIT_PRICE) AS total_revenue
FROM
    DEMO_DW.SALES.FACT_SALES fs
    JOIN DEMO_DW.SALES.DIM_DATE d ON fs.DATE_KEY = d.DATE_KEY
GROUP BY
    d.YEAR,
    d.MONTH
ORDER BY
    d.YEAR,
    d.MONTH;

Result: The analysis shows the monthly revenue trend, allowing observation of changes in sales performance over time.

-- Task 5.1

--Find days where revenue dropped compared to the previous day.

In [ ]:
SELECT
    d.FULL_DATE,
    SUM(fs.QTY * fs.UNIT_PRICE) AS daily_revenue,
    LAG(SUM(fs.QTY * fs.UNIT_PRICE)) OVER (
        ORDER BY
            d.FULL_DATE
    ) AS prev_day_revenue,
    SUM(fs.QTY * fs.UNIT_PRICE) - LAG(SUM(fs.QTY * fs.UNIT_PRICE)) OVER (
        ORDER BY
            d.FULL_DATE
    ) AS revenue_change
FROM
    DEMO_DW.SALES.FACT_SALES fs
    JOIN DEMO_DW.SALES.DIM_DATE d ON fs.DATE_KEY = d.DATE_KEY
GROUP BY
    d.FULL_DATE

Result: Days with revenue decrease were successfully identified using LAG().

-- Task 5.2

-- Filter days where revenue decreased

In [ ]:
SELECT
    c.FIRST_NAME || ' ' || c.LAST_NAME AS customer_name,
    SUM(fs.QTY * fs.UNIT_PRICE) AS total_revenue,
    RANK() OVER (
        ORDER BY
            SUM(fs.QTY * fs.UNIT_PRICE) DESC
    ) AS customer_rank
FROM
    DEMO_DW.SALES.FACT_SALES fs
    JOIN DEMO_DW.SALES.DIM_CUSTOMER c ON fs.CUSTOMER_KEY = c.CUSTOMER_KEY
GROUP BY
    c.FIRST_NAME,
    c.LAST_NAME;

Result: Top customers by revenue contribution were successfully identified and ranked.